# Full-Duplex-Bench Vietnamese — Data Generation Preview

Notebook này chỉ chạy và xem kết quả phần tạo dữ liệu `v1.0` và `v1.5`. Nó không clone repo, không chạy evaluator, không gọi agent/judge. Mặc định output được ghi vào thư mục preview riêng để tránh ghi đè dataset chính.

## 1. Setup đường dẫn và cấu hình preview

- `MAX_SAMPLES`: số sample lấy từ mỗi template để chạy nhanh.
- `OUTPUT_BASE`: thư mục preview. Có thể đổi sang dataset thật nếu cần.
- `BACKGROUND_MODE`: dùng `fake` để xem pipeline Case 4 khi chưa có ElevenLabs/API hoặc local AudioGen/AudioLDM2.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys

from IPython.display import Audio, JSON, Markdown, display


def find_data_generation_dir():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        direct = base / "v1_v1.5" / "data_generation"
        if (direct / "v1_0").exists() and (direct / "v1_5").exists():
            return direct
        if (base / "v1_0").exists() and (base / "v1_5").exists() and (base / "core").exists():
            return base
    raise RuntimeError("Không tìm thấy thư mục v1_v1.5/data_generation từ current working directory.")


DATA_GEN_DIR = find_data_generation_dir()
if str(DATA_GEN_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_GEN_DIR))

OUTPUT_BASE = Path(os.getenv("FDB_PREVIEW_OUTPUT", "/private/tmp/fdb_data_preview")).expanduser()
V1_OUTPUT = OUTPUT_BASE / "v1_0"
V15_OUTPUT = OUTPUT_BASE / "v1_5"

MAX_SAMPLES = 1
RESET_OUTPUT = True
TTS_SEED = int(os.getenv("FDB_TTS_SEED", "20260625"))

# fake: không gọi model tạo noise thật, chỉ kiểm tra được pipeline mix/metadata.
# real: dùng ELEVENLABS_API_KEY hoặc AUDIOGEN_CMD/AUDIOLDM2_CMD theo cấu hình trong template/env.
BACKGROUND_MODE = "fake"  # "fake" hoặc "real"

print("DATA_GEN_DIR:", DATA_GEN_DIR)
print("OUTPUT_BASE:", OUTPUT_BASE)
print("MAX_SAMPLES:", MAX_SAMPLES)
print("BACKGROUND_MODE:", BACKGROUND_MODE)


## 2. Import các hàm tạo data

Cell này dùng trực tiếp code trong repo: `generate_v1_0.py`, `generate_v1_5.py`, `core/tts_generator.py`, `core/audio_mixer.py`.

In [ ]:
from pydub import AudioSegment
from pydub.generators import Sine

from core.audio_mixer import AudioMixer
from core.tts_generator import VietnameseTTSGenerator

from v1_0.generate_v1_0 import (
    generate_pause_handling,
    generate_turn_taking,
    generate_user_interruption,
)
from v1_5.generate_v1_5 import (
    generate_background_speech,
    generate_interruption_and_backchannel,
    generate_talking_to_other,
)


class FakeBackgroundGenerator:
    """Dùng để preview Case 4 mà không cần gọi ElevenLabs/AudioGen/AudioLDM2."""

    def generate(self, prompt, output_path, duration_sec, seed=None, provider=None):
        duration_ms = int(float(duration_sec) * 1000)
        base = Sine(220).to_audio_segment(duration=duration_ms).set_frame_rate(16000).set_channels(1)
        texture = Sine(880).to_audio_segment(duration=duration_ms).set_frame_rate(16000).set_channels(1).apply_gain(-14)
        sound = base.overlay(texture).apply_gain(-16)
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        sound.export(output_path, format="wav", codec="pcm_s16le")
        return {
            "provider": "fake_background_preview",
            "prompt": prompt,
            "duration_sec": duration_sec,
            "seed": seed,
            "note": "Fake tone only for notebook preview; do not use as benchmark noise.",
        }


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def first_n(items, n=MAX_SAMPLES):
    return items[:n] if n else items


def reset_preview_dir(path):
    if RESET_OUTPUT and path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


mixer = AudioMixer()
generator = VietnameseTTSGenerator(provider="edge-tts", seed=TTS_SEED)
print("Imports OK")


## 3. Xem template đầu vào

Cell này giúp kiểm tra nhanh các kịch bản JSON trước khi sinh audio.

In [ ]:
V1_TEMPLATE_FILES = {
    "synthetic_pause_handling": DATA_GEN_DIR / "v1_0" / "templates" / "synthetic_pause_handling.json",
    "candor_turn_taking": DATA_GEN_DIR / "v1_0" / "templates" / "candor_turn_taking.json",
    "synthetic_user_interruption": DATA_GEN_DIR / "v1_0" / "templates" / "synthetic_user_interruption.json",
}

V15_TEMPLATE_FILES = {
    "user_interruption": DATA_GEN_DIR / "v1_5" / "templates" / "user_interruption.json",
    "user_backchannel": DATA_GEN_DIR / "v1_5" / "templates" / "user_backchannel.json",
    "talking_to_other": DATA_GEN_DIR / "v1_5" / "templates" / "talking_to_other.json",
    "background_speech": DATA_GEN_DIR / "v1_5" / "templates" / "background_speech.json",
}

for version, files in [("v1.0", V1_TEMPLATE_FILES), ("v1.5", V15_TEMPLATE_FILES)]:
    display(Markdown(f"### {version}"))
    for name, path in files.items():
        items = read_json(path)
        display(Markdown(f"**{name}** — `{len(items)}` samples — `{path.name}`"))
        display(JSON(items[0] if items else {}))


## 4. Chạy tạo data v1.0

Tạo các task: pause handling, turn taking, user interruption. Output nằm trong `V1_OUTPUT`.

In [ ]:
reset_preview_dir(V1_OUTPUT)

pause_templates = first_n(read_json(V1_TEMPLATE_FILES["synthetic_pause_handling"]))
turn_templates = first_n(read_json(V1_TEMPLATE_FILES["candor_turn_taking"]))
interrupt_templates = first_n(read_json(V1_TEMPLATE_FILES["synthetic_user_interruption"]))

generate_pause_handling(generator, mixer, pause_templates, str(V1_OUTPUT))
generate_turn_taking(generator, mixer, turn_templates, str(V1_OUTPUT))
generate_user_interruption(generator, mixer, interrupt_templates, str(V1_OUTPUT))

print("v1.0 preview output:", V1_OUTPUT)


## 5. Chạy tạo data v1.5

Tạo các task: user interruption, user backchannel, talking to other, background speech. Với `BACKGROUND_MODE = "fake"`, Case 4 không gọi model thật mà chỉ kiểm tra được luồng mix/SNR/metadata.

In [ ]:
reset_preview_dir(V15_OUTPUT)

interrupt_templates = first_n(read_json(V15_TEMPLATE_FILES["user_interruption"]))
backchannel_templates = first_n(read_json(V15_TEMPLATE_FILES["user_backchannel"]))
talk_other_templates = first_n(read_json(V15_TEMPLATE_FILES["talking_to_other"]))
background_templates = first_n(read_json(V15_TEMPLATE_FILES["background_speech"]))

generate_interruption_and_backchannel(generator, mixer, interrupt_templates, "user_interruption", str(V15_OUTPUT))
generate_interruption_and_backchannel(generator, mixer, backchannel_templates, "user_backchannel", str(V15_OUTPUT))
generate_talking_to_other(generator, mixer, talk_other_templates, str(V15_OUTPUT))

background_generator = FakeBackgroundGenerator() if BACKGROUND_MODE == "fake" else None
generate_background_speech(
    generator,
    mixer,
    background_templates,
    str(V15_OUTPUT),
    background_generator=background_generator,
)

print("v1.5 preview output:", V15_OUTPUT)


## 6. Helper xem kết quả

Các helper dưới đây liệt kê sample, phát audio, và in metadata ngay trong notebook.

In [ ]:
def sample_dirs(dataset_root):
    dataset_root = Path(dataset_root)
    if not dataset_root.exists():
        return []
    dirs = []
    for task_dir in sorted(p for p in dataset_root.iterdir() if p.is_dir()):
        dirs.extend(sorted(p for p in task_dir.iterdir() if p.is_dir()))
    return dirs


def audio_info(path):
    sound = AudioSegment.from_file(path)
    return {
        "file": str(path),
        "duration_sec": round(len(sound) / 1000, 3),
        "frame_rate": sound.frame_rate,
        "channels": sound.channels,
        "dBFS": None if sound.dBFS == float("-inf") else round(sound.dBFS, 2),
    }


def show_sample(sample_dir):
    sample_dir = Path(sample_dir)
    display(Markdown(f"### `{sample_dir.parent.name}/{sample_dir.name}`"))

    for audio_name in ["input.wav", "clean_input.wav", "context.wav", "interrupt.wav"]:
        audio_path = sample_dir / audio_name
        if audio_path.exists():
            display(Markdown(f"**{audio_name}**"))
            display(JSON(audio_info(audio_path)))
            display(Audio(filename=str(audio_path)))

    for json_name in ["metadata.json", "pause.json", "turn_taking.json", "interrupt.json"]:
        json_path = sample_dir / json_name
        if json_path.exists():
            display(Markdown(f"**{json_name}**"))
            display(JSON(read_json(json_path)))


def show_dataset(dataset_root, limit=8):
    dirs = sample_dirs(dataset_root)
    display(Markdown(f"Found `{len(dirs)}` samples under `{dataset_root}`"))
    for i, sample_dir in enumerate(dirs[:limit]):
        print(f"[{i}] {sample_dir.parent.name}/{sample_dir.name}")
    return dirs


## 7. Xem kết quả v1.0

In [ ]:
v1_samples = show_dataset(V1_OUTPUT)
if v1_samples:
    show_sample(v1_samples[0])


## 8. Xem kết quả v1.5

In [ ]:
v15_samples = show_dataset(V15_OUTPUT)
if v15_samples:
    show_sample(v15_samples[0])


## 9. Chọn sample bất kỳ để nghe lại

Đổi `SAMPLE_INDEX` rồi chạy cell để xem sample khác.

In [ ]:
ALL_SAMPLES = sample_dirs(V1_OUTPUT) + sample_dirs(V15_OUTPUT)
SAMPLE_INDEX = 0

for i, sample_dir in enumerate(ALL_SAMPLES):
    print(f"[{i}] {sample_dir}")

if ALL_SAMPLES:
    show_sample(ALL_SAMPLES[SAMPLE_INDEX])
